# TORGO Sentences EDA

Exploratory data analysis of the TORGO dataset (sentence-level utterances).

This notebook handles both cases:
1. **Data present**: When TORGO data has been downloaded and manifest built
2. **Data not present**: Uses mock data for testing and development

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

sns.set_style('whitegrid')
%matplotlib inline

# Set random seed for reproducibility
np.random.seed(6140)

## Check Data Availability

Determine if TORGO data is available or if we should use mock data.

In [ ]:
# Check if manifest exists
MANIFEST_PATH = '../manifests/torgo_sentences.csv'
DATA_AVAILABLE = Path(MANIFEST_PATH).exists()

if DATA_AVAILABLE:
    print(f"✓ TORGO data manifest found at {MANIFEST_PATH}")
    print("Running analysis on real data.")
else:
    print(f"✗ Manifest not found at {MANIFEST_PATH}")
    print("Running with MOCK DATA for testing.")
    print("\nTo use real data:")
    print("1. Download TORGO dataset from https://www.cs.toronto.edu/~complingweb/data/TORGO/torgo.html")
    print("2. Extract to ../data/torgo_raw/")
    print("3. Run: python ../scripts/build_torgo_manifest.py")

## Load or Generate Data

Load real manifest if available, otherwise generate realistic mock data for testing.

In [ ]:
def generate_mock_data(n_samples=500, n_speakers=8, n_sessions_per_speaker=3):
    """
    Generate realistic mock TORGO data for testing.
    
    Mimics the structure of real TORGO data:
    - Mix of dysarthric (F/M + number) and control speakers
    - Variable utterance counts per speaker
    - Realistic duration distributions (1-10 seconds)
    - Some missing transcripts (simulating real data issues)
    """
    data = []
    
    # Speaker IDs (TORGO-like: F/M for gender + number)
    speaker_ids = [f"F{i:02d}" for i in range(1, n_speakers//2 + 1)] + \
                  [f"M{i:02d}" for i in range(1, n_speakers//2 + 1)]
    
    utterance_count = 0
    for speaker in speaker_ids:
        # Variable sessions per speaker
        n_sessions = np.random.randint(1, n_sessions_per_speaker + 1)
        
        for session_num in range(1, n_sessions + 1):
            session = f"Session{session_num}"
            
            # Variable utterances per session (10-50)
            n_utterances = np.random.randint(10, 51)
            
            for _ in range(n_utterances):
                utterance_count += 1
                utt_id = f"{utterance_count:04d}"
                
                # Realistic duration: dysarthric speakers tend to have longer utterances
                base_duration = np.random.gamma(2, 2) + 1  # Mean ~5s
                duration = min(max(base_duration, 0.5), 15)  # Clip to 0.5-15s
                
                # Most have transcripts, some don't
                has_transcript = np.random.random() > 0.05
                text = f"This is mock utterance {utt_id} for speaker {speaker}" if has_transcript else None
                
                data.append({
                    'speaker_id': speaker,
                    'session': session,
                    'utt_id': utt_id,
                    'path': f'../data/torgo_raw/{speaker}/{session}/wav_headMic/{utt_id}.wav',
                    'duration': round(duration, 2),
                    'text': text
                })
    
    return pd.DataFrame(data)

# Load or generate data
if DATA_AVAILABLE:
    manifest = pd.read_csv(MANIFEST_PATH)
    # Convert duration to numeric, handling any NA values
    manifest['duration'] = pd.to_numeric(manifest['duration'], errors='coerce')
else:
    print("\n=== GENERATING MOCK DATA ===")
    manifest = generate_mock_data(n_samples=500, n_speakers=8)

print(f"\nTotal utterances: {len(manifest)}")
print(f"Speakers: {manifest['speaker_id'].nunique()}")
print(f"\nFirst few rows:")
manifest.head(10)

## Speaker Distribution

In [ ]:
speaker_counts = manifest['speaker_id'].value_counts().sort_index()
print(f"Number of speakers: {len(speaker_counts)}")
print("\nUtterances per speaker:")
print(speaker_counts)

# Identify speaker types (TORGO convention)
dysarthric_speakers = [s for s in speaker_counts.index if s[0] in ['F', 'M'] and len(s) == 3]
print(f"\nDysarthric speakers: {len(dysarthric_speakers)}")
print(f"Control speakers: {len(speaker_counts) - len(dysarthric_speakers)}")

In [ ]:
# Plot speaker distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
speaker_counts.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Number of Utterances per Speaker', fontsize=12)
axes[0].set_xlabel('Speaker ID')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Pie chart of speaker types
speaker_type_counts = pd.Series({
    'Dysarthric': len(dysarthric_speakers),
    'Control': len(speaker_counts) - len(dysarthric_speakers)
})
if speaker_type_counts.sum() > 0:
    axes[1].pie(speaker_type_counts, labels=speaker_type_counts.index, autopct='%1.0f%%',
                colors=['lightcoral', 'lightblue'])
    axes[1].set_title('Speaker Type Distribution', fontsize=12)

plt.tight_layout()
plt.show()

## Duration Statistics

In [ ]:
if manifest['duration'].notna().any():
    print("Duration statistics:")
    print(manifest['duration'].describe())
    
    total_hours = manifest['duration'].sum() / 3600
    print(f"\nTotal audio duration: {total_hours:.2f} hours")
    
    # Duration by speaker
    print("\nDuration by speaker:")
    speaker_duration = manifest.groupby('speaker_id')['duration'].agg(['sum', 'mean', 'count'])
    speaker_duration['sum_hours'] = speaker_duration['sum'] / 3600
    speaker_duration.columns = ['Total (s)', 'Mean (s)', 'Count', 'Total (hours)']
    print(speaker_duration.sort_values('Total (hours)', ascending=False))
else:
    print("Duration not available in manifest. Run build_torgo_manifest.py with audio files to compute.")

In [ ]:
if manifest['duration'].notna().any():
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Overall duration distribution
    axes[0, 0].hist(manifest['duration'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0, 0].set_title('Overall Duration Distribution')
    axes[0, 0].set_xlabel('Duration (seconds)')
    axes[0, 0].set_ylabel('Count')
    axes[0, 0].axvline(manifest['duration'].mean(), color='red', linestyle='--', 
                       label=f"Mean: {manifest['duration'].mean():.2f}s")
    axes[0, 0].legend()
    
    # Box plot by speaker
    speaker_order = manifest.groupby('speaker_id')['duration'].sum().sort_values(ascending=False).index
    manifest_ordered = manifest.copy()
    manifest_ordered['speaker_id'] = pd.Categorical(
        manifest_ordered['speaker_id'], categories=speaker_order, ordered=True
    )
    sns.boxplot(data=manifest_ordered, x='speaker_id', y='duration', ax=axes[0, 1])
    axes[0, 1].set_title('Duration Distribution by Speaker')
    axes[0, 1].set_xlabel('Speaker ID')
    axes[0, 1].set_ylabel('Duration (seconds)')
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    # Duration vs utterance count scatter
    speaker_stats = manifest.groupby('speaker_id').agg({
        'duration': ['sum', 'count']
    }).reset_index()
    speaker_stats.columns = ['speaker_id', 'total_duration', 'utt_count']
    axes[1, 0].scatter(speaker_stats['utt_count'], speaker_stats['total_duration'], 
                       s=100, alpha=0.6, color='green')
    axes[1, 0].set_xlabel('Number of Utterances')
    axes[1, 0].set_ylabel('Total Duration (seconds)')
    axes[1, 0].set_title('Total Duration vs Utterance Count')
    
    # Cumulative duration
    sorted_durations = manifest['duration'].sort_values(ascending=False).values
    cumulative = np.cumsum(sorted_durations) / 3600
    axes[1, 1].plot(cumulative, color='purple')
    axes[1, 1].set_xlabel('Utterance Rank (sorted by duration)')
    axes[1, 1].set_ylabel('Cumulative Duration (hours)')
    axes[1, 1].set_title('Cumulative Audio Duration')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Session Coverage

In [ ]:
# Session analysis
session_counts = manifest.groupby(['speaker_id', 'session']).size().unstack(fill_value=0)
print("Utterances by Speaker and Session:")
print(session_counts)

print(f"\nTotal sessions: {manifest.groupby(['speaker_id', 'session']).ngroups}")
print(f"Sessions per speaker (mean ± std): {session_counts.sum(axis=1).mean():.1f} ± {session_counts.sum(axis=1).std():.1f}")

In [ ]:
# Session heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(session_counts, annot=True, fmt='d', cmap='YlOrRd', cbar_kws={'label': 'Utterances'})
plt.title('Utterances by Speaker and Session', fontsize=14)
plt.xlabel('Session')
plt.ylabel('Speaker ID')
plt.tight_layout()
plt.show()

## Missing Data Analysis

In [ ]:
print("=== MISSING DATA ANALYSIS ===\n")

# Check for missing files
if DATA_AVAILABLE:
    missing = []
    for _, row in manifest.iterrows():
        if not Path(row['path']).exists():
            missing.append(row['path'])
    
    print(f"Missing audio files: {len(missing)}")
    if missing:
        print("Examples:", missing[:5])
else:
    print("(Skipping file existence check in mock data mode)")

# Check for missing durations
missing_duration = manifest['duration'].isna().sum()
print(f"\nMissing durations: {missing_duration} / {len(manifest)} ({100*missing_duration/len(manifest):.1f}%)")

# Check for missing transcripts
missing_transcript = manifest['text'].isna().sum()
print(f"Missing transcripts: {missing_transcript} / {len(manifest)} ({100*missing_transcript/len(manifest):.1f}%)")

# Missing transcript by speaker
if missing_transcript > 0:
    print("\nMissing transcripts by speaker:")
    missing_by_speaker = manifest[manifest['text'].isna()].groupby('speaker_id').size()
    print(missing_by_speaker)

## Transcript Analysis (if available)

In [ ]:
if manifest['text'].notna().any():
    # Text length analysis
    manifest_with_text = manifest[manifest['text'].notna()].copy()
    manifest_with_text['text_length'] = manifest_with_text['text'].str.len()
    manifest_with_text['word_count'] = manifest_with_text['text'].str.split().str.len()
    
    print("Text statistics:")
    print(f"  Mean characters: {manifest_with_text['text_length'].mean():.1f}")
    print(f"  Mean words: {manifest_with_text['word_count'].mean():.1f}")
    
    # Correlation between duration and word count
    corr = manifest_with_text['duration'].corr(manifest_with_text['word_count'])
    print(f"\nDuration-WordCount correlation: {corr:.3f}")
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(manifest_with_text['word_count'], bins=20, color='steelblue', edgecolor='black')
    axes[0].set_xlabel('Word Count')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Transcript Word Counts')
    
    axes[1].scatter(manifest_with_text['word_count'], manifest_with_text['duration'], 
                    alpha=0.5, color='purple')
    axes[1].set_xlabel('Word Count')
    axes[1].set_ylabel('Duration (seconds)')
    axes[1].set_title(f'Duration vs Word Count (r={corr:.2f})')
    
    plt.tight_layout()
    plt.show()
else:
    print("No transcript data available.")

## Pilot Subset Selection

Select 2-3 speakers with the most data for the Week 1 pilot.
Criteria:
- Prioritize speakers with most utterances
- Consider duration coverage
- Include mix of speaker types if possible

In [ ]:
# Calculate speaker statistics for selection
speaker_stats = manifest.groupby('speaker_id').agg({
    'utt_id': 'count',
    'duration': 'sum',
    'text': lambda x: x.notna().sum()  # Count non-null transcripts
}).reset_index()
speaker_stats.columns = ['speaker_id', 'utterance_count', 'total_duration', 'transcript_count']
speaker_stats['duration_hours'] = speaker_stats['total_duration'] / 3600
speaker_stats['transcript_coverage'] = 100 * speaker_stats['transcript_count'] / speaker_stats['utterance_count']

# Sort by utterance count (primary) and duration (secondary)
speaker_stats = speaker_stats.sort_values(
    ['utterance_count', 'total_duration'], ascending=False
)

print("Speaker Statistics (sorted by utterance count):")
print(speaker_stats.to_string(index=False))

# Select top 3 speakers
N_PILOT_SPEAKERS = 3
pilot_speakers = speaker_stats.head(N_PILOT_SPEAKERS)['speaker_id'].tolist()

print(f"\n{'='*50}")
print(f"SELECTED PILOT SPEAKERS: {pilot_speakers}")
print(f"{'='*50}")

# Pilot subset stats
pilot_manifest = manifest[manifest['speaker_id'].isin(pilot_speakers)].copy()
pilot_duration_hours = pilot_manifest['duration'].sum() / 3600

print(f"\nPilot Subset Statistics:")
print(f"  Utterances: {len(pilot_manifest)} / {len(manifest)} ({100*len(pilot_manifest)/len(manifest):.1f}%)")
print(f"  Duration: {pilot_duration_hours:.2f} hours")
print(f"  Speakers: {len(pilot_speakers)}")
print(f"  Transcript coverage: {100*pilot_manifest['text'].notna().sum()/len(pilot_manifest):.1f}%")

print(f"\nBreakdown by speaker:")
for speaker in pilot_speakers:
    s_data = pilot_manifest[pilot_manifest['speaker_id'] == speaker]
    s_duration = s_data['duration'].sum() / 3600
    print(f"  {speaker}: {len(s_data)} utterances, {s_duration:.2f} hours")

In [ ]:
# Visualize pilot selection
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Highlight pilot speakers
colors = ['red' if s in pilot_speakers else 'lightblue' for s in speaker_stats['speaker_id']]
axes[0].bar(speaker_stats['speaker_id'], speaker_stats['utterance_count'], color=colors)
axes[0].set_xlabel('Speaker ID')
axes[0].set_ylabel('Utterance Count')
axes[0].set_title('Pilot Speaker Selection (red = selected)')
axes[0].tick_params(axis='x', rotation=45)

# Pie chart of pilot data contribution
pilot_counts = speaker_stats.set_index('speaker_id')['utterance_count']
other_count = pilot_counts.drop(pilot_speakers).sum()
pie_data = pilot_counts[pilot_speakers].tolist() + [other_count]
pie_labels = pilot_speakers + ['Others']
pie_colors = ['red', 'darkred', 'crimson', 'lightblue']

axes[1].pie(pie_data, labels=pie_labels, autopct='%1.1f%%', colors=pie_colors)
axes[1].set_title('Data Distribution: Pilot vs Others')

plt.tight_layout()
plt.show()

In [ ]:
# Save pilot manifest (only if using real data)
if DATA_AVAILABLE:
    pilot_output_path = '../manifests/torgo_pilot.csv'
    pilot_manifest.to_csv(pilot_output_path, index=False)
    print(f"✓ Saved pilot manifest to {pilot_output_path}")
else:
    print("(Mock data mode - not saving pilot manifest)")
    print("\nWhen using real data, pilot manifest would be saved to: ../manifests/torgo_pilot.csv")

## Summary for Week 1

This analysis provides:
1. **Speaker coverage**: Understanding data distribution across speakers
2. **Duration analysis**: Total audio hours and per-speaker breakdown
3. **Session coverage**: Multi-session availability for train/test splits
4. **Data quality**: Missing files, transcripts, durations
5. **Pilot selection**: 2-3 speakers with most data for initial experiments

### Next Steps
1. ✓ Select pilot speakers (completed above)
2. Generate teacher labels using Silero VAD
3. Extract features (mel spectrograms)
4. Train student model on pilot subset